# Qualifying Session
- **Q1**: Bottom 14 from Practice → top 2 advance to Q2
- **Q2**: Top 10 from Practice + 2 from Q1 → determines starting grid (P1–P12)
- Session time: 15 min (900s) → laps = floor(900 / base_lap_time) − 1
- Push mode: last 3 laps

In [7]:
import pandas as pd
import numpy as np
import time
from pathlib import Path

RAW       = Path('../data/raw')
PUSH_LAPS = 3
LAP_DELAY = 2
MAX_DELTA = 3.0

In [8]:
entry_info    = pd.read_csv(RAW / 'entry_info.csv')
riders_rating = pd.read_csv(RAW / 'riders_rating.csv').rename(columns={
    'braking': 'rider_braking', 'cornering': 'rider_cornering'
})
bikes_rating  = pd.read_csv(RAW / 'bikes_rating.csv').rename(columns={
    'braking': 'bike_braking', 'cornering': 'bike_cornering'
})
circuits = pd.read_csv(RAW / 'circuits.csv')

df = (
    entry_info
    .merge(riders_rating, on='name', how='left')
    .merge(bikes_rating,  on=['manufacturer', 'team_status'], how='left')
)

# ── Circuit selection (must match Practice notebook) ──
CIRCUIT_IDX = 0
circuit    = circuits.iloc[CIRCUIT_IDX]
country    = circuit['country']
report_dir = Path('../report') / country

total_laps    = int(900 // circuit['base_lap_time']) - 1
push_from_lap = total_laps - PUSH_LAPS + 1

print(f"Circuit    : {circuit['circuit_name']} ({country})")
print(f"Base lap   : {circuit['base_lap_time']}s")
print(f"Total laps : {total_laps}  |  Push mode from lap: {push_from_lap}")

# ── Parse Practice classification from Practice.md ──
practice_md = (report_dir / 'Practice.md').read_text(encoding='utf-8')
md_rows = [l for l in practice_md.split('\n') if l.startswith('| P') and not l.startswith('| P |')]
practice_order = []
for line in md_rows:
    parts = [p.strip() for p in line.split('|')]
    practice_order.append({'position': int(parts[1][1:]), 'name': parts[3]})
practice_df = pd.DataFrame(practice_order)

top10_names = set(practice_df[practice_df['position'] <= 10]['name'])
print(f"\nTop 10 from Practice (→ Q2 direct): {len(top10_names)} riders")
print(f"Q1 riders: {24 - len(top10_names)}")

Circuit    : Liam Henderson Circuit (Australia)
Base lap   : 87s
Total laps : 9  |  Push mode from lap: 7

Top 10 from Practice (→ Q2 direct): 10 riders
Q1 riders: 14


In [9]:
# ── Engine ───────────────────────────────────────────────────────────────────

def norm(v):
    return (v - 70) / 29

def circuit_weights(c):
    sr = c['straight_length_m'] / (c['lap_length_km'] * 1000)
    cd = c['corners'] / c['lap_length_km']
    w_spd, w_cor, w_brk = sr * 3, cd / 5, 0.30
    t = w_spd + w_cor + w_brk
    return w_spd/t, w_cor/t, w_brk/t

def perf_score_quali(row, w_spd, w_cor, w_brk):
    bike = (
        w_spd * (norm(row['top_speed']) * 0.6 + norm(row['acceleration']) * 0.4)
        + w_cor * norm(row['bike_cornering'])
        + w_brk * norm(row['bike_braking'])
    )
    # In qualifying aggression carries more weight
    rw_cor = w_cor + w_spd * 0.2
    rw_brk = w_brk + w_spd * 0.3
    rw_agg = w_spd * 0.5
    rw_tot = rw_cor + rw_brk + rw_agg
    rider = (
        (rw_cor / rw_tot) * norm(row['rider_cornering'])
        + (rw_brk / rw_tot) * norm(row['rider_braking'])
        + (rw_agg / rw_tot) * norm(row['aggression'])
    )
    return 0.55 * bike + 0.45 * rider

def simulate_quali_lap(row, base_time, lap_num, score, is_push):
    if is_push:
        # Crash risk: aggressive + inconsistent riders crash more
        crash_prob = 0.05 + 0.06 * norm(row['aggression']) * (1 - norm(row['consistency']))
        if np.random.random() < crash_prob:
            return None  # CRASH
        push_gain = 0.3 + 0.2 * norm(row['aggression'])
        time_pen  = max(0.0, (1 - score) * MAX_DELTA - push_gain)
        variance  = 0.8 * (1 - norm(row['consistency']) * 0.5)
    else:
        time_pen = (1 - score) * MAX_DELTA
        variance = 0.5 * (1 - norm(row['consistency'])) * (1 - norm(row['stability']))

    warmup  = 0.5 if lap_num == 1 else 0.0
    noise   = np.random.uniform(-variance, variance)
    return max(base_time + time_pen + warmup + noise, base_time * 0.97)

def fmt_lap(s):
    m = int(s // 60)
    return f'{m:02d}:{s % 60:06.3f}'

def fmt_gap(g):
    return '—' if g == 0 else f'+{g:.3f}'

def run_session(riders_subset, session_label):
    """Run a qualifying session and return the classification DataFrame."""
    base_time        = circuit['base_lap_time']
    w_spd, w_cor, w_brk = circuit_weights(circuit)
    results          = []

    print('=' * 68)
    print(f'  {session_label}')
    print(f"  {circuit['circuit_name'].upper()}  —  {country.upper()}")
    print(f'  Laps: {total_laps}  |  Push mode: last {PUSH_LAPS} laps (lap {push_from_lap}+)')
    print('=' * 68)

    for _, rider in riders_subset.sort_values('bike_number').iterrows():
        score     = perf_score_quali(rider, w_spd, w_cor, w_brk)
        lap_times = []

        print(f"\n  #{rider['bike_number']:02d}  {rider['name']}  |  {rider['team']}  [{rider['team_status'].upper()}]")
        print(f"  {'LAP':>5}   {'TIME':>10}   {'BEST':>10}")
        print(f"  {'─'*35}")

        for lap_num in range(1, total_laps + 1):
            is_push   = lap_num >= push_from_lap
            lap_sec   = simulate_quali_lap(rider, base_time, lap_num, score, is_push)
            prev_best = min(lap_times) if lap_times else float('inf')
            push_tag  = '  [PUSH]' if is_push else ''

            if lap_sec is not None:
                lap_times.append(lap_sec)
                best   = min(lap_times)
                pb_tag = '  ◄ PB' if best < prev_best else ''
                print(f"  Lap {lap_num:2d}   {fmt_lap(lap_sec):>10}   {fmt_lap(best):>10}{pb_tag}{push_tag}")
            else:
                best_str = fmt_lap(min(lap_times)) if lap_times else 'NO TIME'
                print(f"  Lap {lap_num:2d}   {'CRASH':>10}   {best_str:>10}{push_tag}")

        best_sec = min(lap_times) if lap_times else float('inf')
        results.append({
            'bike_number' : int(rider['bike_number']),
            'name'        : rider['name'],
            'team'        : rider['team'],
            'team_status' : rider['team_status'],
            'manufacturer': rider['manufacturer'],
            'best_lap_sec': best_sec,
            'best_lap'    : fmt_lap(best_sec) if best_sec != float('inf') else 'NO TIME',
        })

    print('\n' + '=' * 68)

    class_df = (
        pd.DataFrame(results)
        .sort_values('best_lap_sec')
        .reset_index(drop=True)
    )
    class_df.index += 1
    class_df['gap']     = class_df['best_lap_sec'] - class_df['best_lap_sec'].iloc[0]
    class_df['gap_fmt'] = class_df['gap'].apply(fmt_gap)
    return class_df

print('Engine ready.')

Engine ready.


In [10]:
# ── Q1 Session ───────────────────────────────────────────────────────────────

q1_riders = df[~df['name'].isin(top10_names)].copy()
q1_class  = run_session(q1_riders, 'QUALIFYING 1')

# ── Q1 Classification ────────────────────────────────────────────────────────
print(f"\n{'─'*68}")
print( '  Q1 — CLASSIFICATION')
print(f"{'─'*68}")
print(f"  {'P':<4} {'#':<5} {'RIDER':<24} {'TEAM':<26} {'BEST LAP':>10} {'GAP':>8}")
print(f"  {'─'*66}")
for pos, row in q1_class.iterrows():
    advance = '  ★ ADVANCES TO Q2' if pos <= 2 else ''
    print(f"  P{pos:<3} #{row['bike_number']:<4} {row['name']:<24} {row['team']:<26} {row['best_lap']:>10} {row['gap_fmt']:>8}{advance}")

q2_advance_names = set(q1_class.head(2)['name'])
q1_non_q2        = q1_class.iloc[2:].copy()

print(f"\n{'='*68}")
print(f"  ★ Advancing to Q2: {', '.join(q2_advance_names)}")
print(f"{'='*68}")

  QUALIFYING 1
  LIAM HENDERSON CIRCUIT  —  AUSTRALIA
  Laps: 9  |  Push mode: last 3 laps (lap 7+)

  #10  Victor Burgos  |  Inferno Factory  [SATELLITE]
    LAP         TIME         BEST
  ───────────────────────────────────
  Lap  1    01:29.690    01:29.690  ◄ PB
  Lap  2    01:28.873    01:28.873  ◄ PB
  Lap  3    01:29.148    01:28.873
  Lap  4    01:29.053    01:28.873
  Lap  5    01:29.031    01:28.873
  Lap  6    01:29.054    01:28.873
  Lap  7    01:28.544    01:28.544  ◄ PB  [PUSH]
  Lap  8    01:28.947    01:28.544  [PUSH]
  Lap  9    01:28.245    01:28.245  ◄ PB  [PUSH]

  #12  Francesco Carelli  |  Phoenix Motorsport  [SATELLITE]
    LAP         TIME         BEST
  ───────────────────────────────────
  Lap  1    01:29.846    01:29.846  ◄ PB
  Lap  2    01:29.349    01:29.349  ◄ PB
  Lap  3    01:29.319    01:29.319  ◄ PB
  Lap  4    01:29.354    01:29.319
  Lap  5    01:29.180    01:29.180  ◄ PB
  Lap  6    01:29.358    01:29.180
  Lap  7    01:28.617    01:28.617  ◄ PB  

In [11]:
# ── Q2 Session ───────────────────────────────────────────────────────────────

q2_riders = pd.concat([
    df[df['name'].isin(top10_names)],
    df[df['name'].isin(q2_advance_names)]
], ignore_index=True)

q2_class = run_session(q2_riders, 'QUALIFYING 2')

# ── Q2 Classification ────────────────────────────────────────────────────────
print(f"\n{'─'*82}")
print( '  Q2 — CLASSIFICATION  (= STARTING GRID P1–P12)')
print(f"{'─'*82}")
print(f"  {'P':<4} {'#':<5} {'RIDER':<24} {'TEAM':<26} {'MANUFACTURER':<14} {'BEST LAP':>10} {'GAP':>8}")
print(f"  {'─'*78}")
for pos, row in q2_class.iterrows():
    pole = '  ◀ POLE' if pos == 1 else ''
    print(f"  P{pos:<3} #{row['bike_number']:<4} {row['name']:<24} {row['team']:<26} {row['manufacturer']:<14} {row['best_lap']:>10} {row['gap_fmt']:>8}{pole}")
print(f"{'─'*82}")

  QUALIFYING 2
  LIAM HENDERSON CIRCUIT  —  AUSTRALIA
  Laps: 9  |  Push mode: last 3 laps (lap 7+)

  #07  Petros Georgiou  |  Honda Factory Racing  [FACTORY]
    LAP         TIME         BEST
  ───────────────────────────────────
  Lap  1    01:28.919    01:28.919  ◄ PB
  Lap  2    01:28.424    01:28.424  ◄ PB
  Lap  3    01:28.671    01:28.424
  Lap  4    01:28.535    01:28.424
  Lap  5    01:28.520    01:28.424
  Lap  6    01:28.614    01:28.424
  Lap  7    01:28.719    01:28.424  [PUSH]
  Lap  8    01:28.637    01:28.424  [PUSH]
  Lap  9    01:28.377    01:28.377  ◄ PB  [PUSH]

  #19  Matteo Esposito  |  Honda Factory Racing  [FACTORY]
    LAP         TIME         BEST
  ───────────────────────────────────
  Lap  1    01:28.799    01:28.799  ◄ PB
  Lap  2    01:28.316    01:28.316  ◄ PB
  Lap  3    01:28.306    01:28.306  ◄ PB
  Lap  4    01:28.321    01:28.306
  Lap  5    01:28.301    01:28.301  ◄ PB
  Lap  6    01:28.289    01:28.289  ◄ PB
  Lap  7    01:27.786    01:27.786  ◄ P

In [12]:
# ── Export Q1.md / Q2.md / grid.md ──────────────────────────────────────────

title = f"{circuit['circuit_name']} - GRAND PRIX OF {country.upper()}"

def md_table(class_df):
    header = '| P | # | RIDER | TEAM | MANUFACTURER | BEST LAP | GAP |'
    sep    = '|---|---|-------|------|--------------|----------|-----|'
    rows   = [
        f"| P{pos} | #{r['bike_number']} | {r['name']} | {r['team']} | {r['manufacturer']} | {r['best_lap']} | {r['gap_fmt']} |"
        for pos, r in class_df.iterrows()
    ]
    return f"{header}\n{sep}\n" + "\n".join(rows)

# Q1.md
(report_dir / 'Q1.md').write_text(
    f"# {title}\n\n## Qualifying 1 - Classification\n\n{md_table(q1_class)}\n",
    encoding='utf-8'
)

# Q2.md
(report_dir / 'Q2.md').write_text(
    f"# {title}\n\n## Qualifying 2 - Classification\n\n{md_table(q2_class)}\n",
    encoding='utf-8'
)

# grid.md — Q2 results (P1–P12) | separator | Q1 non-qualifiers (P13–P24)
q1_nq = q1_non_q2.reset_index(drop=True)
q1_nq.index = range(13, 13 + len(q1_nq))
q1_nq['gap']     = q1_nq['best_lap_sec'] - q1_nq['best_lap_sec'].iloc[0]
q1_nq['gap_fmt'] = q1_nq['gap'].apply(fmt_gap)

grid_md = (
    f"# {title}\n\n"
    f"## Starting Grid\n\n"
    f"### Q2 Qualifiers\n\n{md_table(q2_class)}\n\n"
    f"---\n\n"
    f"### Q1 (Non-Qualifiers)\n\n{md_table(q1_nq)}\n"
)
(report_dir / 'grid.md').write_text(grid_md, encoding='utf-8')

print(f"Saved: Q1.md | Q2.md | grid.md  →  {report_dir}")

Saved: Q1.md | Q2.md | grid.md  →  ..\report\Australia
